# Model 3 — VGG16 Transfer Learning

**Group**: Group 14  
**Members**: *(update with real names)*  
**Model owner**: P3  
**Architecture**: VGG16 (ImageNet weights, include_top=False) + GlobalAveragePooling2D + Dense head  
**Dataset**: NIH Malaria Cell Images — Parasitized vs Uninfected  
**Date**: June 2026

---
This notebook applies VGG16 pretrained on ImageNet to malaria cell classification via transfer learning.  
A two-stage training strategy is used: first train only the classification head with the base frozen,  
then fine-tune the top layers of the base at a reduced learning rate.

## 1. Environment Setup
Install dependencies, set all random seeds for reproducibility, and verify GPU availability.

In [1]:
!pip install -q kagglehub tqdm

import os, sys, random
import numpy as np
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU available:', gpus if gpus else 'None — training on CPU')


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


TensorFlow version: 2.21.0
GPU available: None — training on CPU


## 2. Dataset Download
Download the NIH Malaria dataset via `kagglehub`. The path is resolved dynamically so this notebook runs on any machine or Colab session.

In [2]:
import kagglehub

path = kagglehub.dataset_download('iarunava/cell-images-for-detecting-malaria')
print('Downloaded to:', path)

DATA_DIR = None
for root, dirs, _ in os.walk(path):
    if 'Parasitized' in dirs and 'Uninfected' in dirs:
        DATA_DIR = root
        break

assert DATA_DIR is not None, 'Could not locate Parasitized/Uninfected folders'
print('DATA_DIR:', DATA_DIR)
print('Parasitized images:', len(os.listdir(os.path.join(DATA_DIR, 'Parasitized'))))
print('Uninfected images: ', len(os.listdir(os.path.join(DATA_DIR, 'Uninfected'))))

c:\Users\Luqma\dev\projects\coursework\Group14_Formative2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloaded to: C:\Users\Luqma\.cache\kagglehub\datasets\iarunava\cell-images-for-detecting-malaria\versions\1
DATA_DIR: C:\Users\Luqma\.cache\kagglehub\datasets\iarunava\cell-images-for-detecting-malaria\versions\1\cell_images
Parasitized images: 13780
Uninfected images:  13780


## 3. Shared Helper Functions
All helper functions are defined inline — this notebook runs independently on any platform without needing `utils.py`.  
**Note**: VGG16 requires raw pixel values [0, 255] — `vgg16.preprocess_input` is applied inside the model, so `load_dataset` is called with `normalise=False`.

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve,
)

# ── Constants
IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32
SEED        = 42
CLASS_NAMES = ['Parasitized', 'Uninfected']
TRAIN_SPLIT = 0.80
VAL_SPLIT   = 0.10
TEST_SPLIT  = 0.10

# Fraction of full dataset used — same across all 5 models for fair comparison
DATASET_FRACTION = 0.2

os.makedirs('figures',     exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

# ── Augmentation layer
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name='data_augmentation')

# ── Dataset loader
def load_dataset(data_dir, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
                 normalise=False, fraction=1.0):
    """
    fraction: float in (0, 1] — proportion of full dataset to use.
    All 5 models use the same DATASET_FRACTION for fair cross-model comparison.
    normalise: True for custom CNNs (/255); False for pretrained models (preprocess_input inside model).
    """
    full_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        labels='inferred',
        label_mode='binary',
        class_names=CLASS_NAMES,
        image_size=image_size,
        batch_size=None,
        shuffle=True,
        seed=SEED,
    )
    total = sum(1 for _ in full_ds)
    use   = int(total * fraction) if fraction < 1.0 else total
    if fraction < 1.0:
        full_ds = full_ds.take(use)
        print(f'Using {use}/{total} images ({fraction*100:.0f}% of dataset)')

    n_train   = int(use * TRAIN_SPLIT)
    n_val     = int(use * VAL_SPLIT)
    train_ds  = full_ds.take(n_train)
    remaining = full_ds.skip(n_train)
    val_ds    = remaining.take(n_val)
    test_ds   = remaining.skip(n_val)
    AUTOTUNE  = tf.data.AUTOTUNE
    cast_fn = (lambda img, lbl: (tf.cast(img, tf.float32) / 255.0, lbl)) if normalise \
              else (lambda img, lbl: (tf.cast(img, tf.float32), lbl))
    train_ds = (train_ds.map(cast_fn, num_parallel_calls=AUTOTUNE)
                .cache().shuffle(1000, seed=SEED).batch(batch_size).prefetch(AUTOTUNE))
    val_ds   = (val_ds.map(cast_fn, num_parallel_calls=AUTOTUNE)
                .cache().batch(batch_size).prefetch(AUTOTUNE))
    test_ds  = (test_ds.map(cast_fn, num_parallel_calls=AUTOTUNE)
                .cache().batch(batch_size).prefetch(AUTOTUNE))
    return train_ds, val_ds, test_ds

# ── Metrics
def evaluate_model(model, test_ds):
    y_true, y_pred_prob = [], []
    for images, labels in test_ds:
        preds = model(images, training=False).numpy().flatten()
        y_pred_prob.extend(preds)
        y_true.extend(labels.numpy().flatten())
    y_true      = np.array(y_true)
    y_pred_prob = np.array(y_pred_prob)
    y_pred      = (y_pred_prob >= 0.5).astype(int)
    return {
        'accuracy':  round(accuracy_score(y_true, y_pred),     4),
        'precision': round(precision_score(y_true, y_pred),    4),
        'recall':    round(recall_score(y_true, y_pred),       4),
        'f1':        round(f1_score(y_true, y_pred),           4),
        'auc':       round(roc_auc_score(y_true, y_pred_prob), 4),
        'y_true':    y_true,
        'y_pred':    y_pred,
        'y_prob':    y_pred_prob,
    }

# ── Learning curves
def plot_learning_curves(history, experiment_name, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Learning curves — {experiment_name}', fontsize=14, fontweight='bold')
    axes[0].plot(history.history['loss'],     label='Train loss',     linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Val loss',       linewidth=2, linestyle='--')
    axes[0].set_title('Loss over epochs'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(history.history['accuracy'],     label='Train accuracy', linewidth=2)
    axes[1].plot(history.history['val_accuracy'], label='Val accuracy',   linewidth=2, linestyle='--')
    axes[1].set_title('Accuracy over epochs'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    final_train = history.history['accuracy'][-1]
    final_val   = history.history['val_accuracy'][-1]
    gap = final_train - final_val
    if gap > 0.05:
        print(f'Overfitting detected: train {final_train:.3f} vs val {final_val:.3f} (gap={gap:.3f})')
    elif gap < -0.02:
        print(f'Underfitting detected: train {final_train:.3f} vs val {final_val:.3f}')
    else:
        print(f'Good fit: train {final_train:.3f} vs val {final_val:.3f} (gap={gap:.3f})')

# ── Confusion matrix
def plot_confusion_matrix(metrics_dict, class_names, experiment_name, save_path=None):
    cm  = confusion_matrix(metrics_dict['y_true'], metrics_dict['y_pred'])
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax, linewidths=0.5)
    ax.set_title(f'Confusion matrix — {experiment_name}', fontsize=13, fontweight='bold', pad=14)
    ax.set_ylabel('True label', fontsize=11); ax.set_xlabel('Predicted label', fontsize=11)
    tn, fp, fn, tp = cm.ravel()
    ax.text(0.5, -0.12,
            f'TP={tp}  TN={tn}  FP={fp}  FN={fn}  |  Sensitivity={tp/(tp+fn):.3f}  Specificity={tn/(tn+fp):.3f}',
            ha='center', va='top', transform=ax.transAxes, fontsize=10, color='dimgray')
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

# ── ROC curve
def plot_roc_curve(metrics_dict, experiment_name, save_path=None):
    fpr, tpr, _ = roc_curve(metrics_dict['y_true'], metrics_dict['y_prob'])
    auc_val = metrics_dict['auc']
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, color='royalblue', lw=2, label=f'ROC curve (AUC = {auc_val:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
    ax.fill_between(fpr, tpr, alpha=0.08, color='royalblue')
    ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
    ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=11)
    ax.set_title(f'ROC Curve — {experiment_name}', fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

# ── Results table
def build_results_table(experiments_list):
    df = pd.DataFrame(experiments_list)
    df = df[['exp_num','description','accuracy','precision','recall','f1','auc','epochs','notes']]
    df.columns = ['Exp #','Description','Accuracy','Precision','Recall','F1','AUC','Epochs','Notes']
    return df.sort_values('F1', ascending=False).reset_index(drop=True)

# ── Callbacks
def get_callbacks(model_name, experiment_num, patience_es=10, patience_lr=5):
    os.makedirs('checkpoints', exist_ok=True)
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=patience_es, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=patience_lr, min_lr=1e-7, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f'checkpoints/{model_name}_exp{experiment_num}.h5',
            monitor='val_accuracy', save_best_only=True, verbose=0),
    ]

# ── Error analysis
def error_analysis(model, test_ds, class_names, n_samples=12):
    misclassified_images, misclassified_labels, misclassified_preds = [], [], []
    for images, labels in test_ds:
        preds        = model(images, training=False).numpy().flatten()
        pred_classes = (preds >= 0.5).astype(int)
        labels_np    = labels.numpy().astype(int).flatten()
        images_np    = images.numpy()
        mask         = pred_classes != labels_np
        misclassified_images.extend(images_np[mask])
        misclassified_labels.extend(labels_np[mask])
        misclassified_preds.extend(preds[mask])
        if len(misclassified_images) >= n_samples:
            break
    n   = min(n_samples, len(misclassified_images))
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    fig.suptitle('Error Analysis — Misclassified Samples', fontsize=14, fontweight='bold')
    for i, ax in enumerate(axes.flatten()[:n]):
        img = misclassified_images[i]
        if img.max() > 1.0:
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        ax.imshow(img)
        true_lbl = class_names[int(misclassified_labels[i])]
        pred_lbl = class_names[int(misclassified_preds[i] >= 0.5)]
        conf     = misclassified_preds[i] if pred_lbl == class_names[1] else 1 - misclassified_preds[i]
        ax.set_title(f'True: {true_lbl}\nPred: {pred_lbl} ({conf:.2f})', fontsize=9, color='red')
        ax.axis('off')
    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/P3_error_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

print('Helper functions loaded.')


Helper functions loaded.


## 4. Load Dataset
VGG16 requires 224×224 input. Crucially, `normalise=False` is passed so raw [0, 255] pixel values are preserved — `vgg16.preprocess_input` inside the model handles channel-wise mean subtraction as required by the ImageNet-pretrained weights.

> **Computational note**: `DATASET_FRACTION = 0.2` — 20% of the full 27,558-image dataset (~5,511 images) is used across **all 5 models** for consistent cross-model comparison and feasible CPU training time.

In [4]:
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

train_ds, val_ds, test_ds = load_dataset(
    data_dir=DATA_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    normalise=False,  # VGG16 preprocess_input applied inside model
    fraction=DATASET_FRACTION,
)

n_train = sum(1 for _ in train_ds) * BATCH_SIZE
n_val   = sum(1 for _ in val_ds)   * BATCH_SIZE
n_test  = sum(1 for _ in test_ds)  * BATCH_SIZE
print(f'Train: ~{n_train} | Val: ~{n_val} | Test: ~{n_test}')

Found 27558 files belonging to 2 classes.
Train: ~22048 | Val: ~2784 | Test: ~2784


## 5. Model Architecture
VGG16 base (ImageNet weights, top excluded) with a custom classification head.  
`freeze_base=True` freezes all base layers for Stage 1 training.  
`fine_tune_from` unfreezes layers from that index onward for Stage 2 fine-tuning.  
`vgg16.preprocess_input` is applied as the first step inside the model — never use /255 normalisation with pretrained models.

In [5]:
from tensorflow.keras.regularizers import l2

def build_vgg16_model(freeze_base=True, fine_tune_from=None,
                      use_augmentation=False, dense_units=(256, 128),
                      l2_lambda=None):
    """
    freeze_base     : freeze entire VGG16 base
    fine_tune_from  : int index — unfreeze layers from this index onward (negative = from end)
    use_augmentation: prepend data_augmentation layer
    dense_units     : tuple of Dense layer sizes in the head
    l2_lambda       : if set, apply L2 regularisation to Dense layers
    """
    base_model = tf.keras.applications.VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = not freeze_base

    if fine_tune_from is not None:
        base_model.trainable = True
        n = len(base_model.layers)
        cutoff = n + fine_tune_from if fine_tune_from < 0 else fine_tune_from
        for layer in base_model.layers[:cutoff]:
            layer.trainable = False
        trainable_names = [l.name for l in base_model.layers if l.trainable]
        print(f'Fine-tuning {len(trainable_names)} base layers: {trainable_names}')

    reg = l2(l2_lambda) if l2_lambda else None

    inputs = tf.keras.Input(shape=(224, 224, 3))
    x      = inputs

    if use_augmentation:
        x = data_augmentation(x)

    x = tf.keras.applications.vgg16.preprocess_input(x)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    for units in dense_units:
        x = tf.keras.layers.Dense(units, activation='relu',
                                  kernel_regularizer=reg)(x)
        x = tf.keras.layers.Dropout(0.5 if units >= 256 else 0.3)(x)

    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model   = tf.keras.Model(inputs, outputs, name='vgg16_transfer')
    return model, base_model

# Preview default architecture (frozen base)
m, _ = build_vgg16_model(freeze_base=True)
m.summary()
del m

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "vgg16_transfer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 224, 224)  │          0 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 224, 224)  │          0 │ input_layer_1[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 224, 224)  │          0 │ input_layer_1[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 224, 224,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 224, 224,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg16 (Functional)  │ (None, 7, 7, 512) │ 14,714,688 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 512)       │          0 │ vgg16[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    131,328 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │        129 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,879,041 (56.76 MB)

 Trainable params: 164,353 (642.00 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

## 6. Experiment Tracking
All experiment results are appended to `results_log`. The final summary table is displayed after all 7 experiments.

In [6]:
results_log = []  # Append one dict per experiment — never overwrite

---
## Experiment 1: Fully Frozen Base — Head Only

**Hypothesis**: Training only the classification head while keeping all VGG16 layers frozen should rapidly converge to a strong baseline, leveraging the rich ImageNet feature representations without any risk of destroying pretrained weights.

**Change made**: `freeze_base=True`, head=Dense(256)+Dense(128), 10 epochs

In [7]:
EXP_NUM         = 1
EXP_DESCRIPTION = 'Fully frozen base — head only (Dense 256+128)'
EPOCHS          = 10

model1, base1 = build_vgg16_model(freeze_base=True)
model1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history1 = model1.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=get_callbacks('vgg16', EXP_NUM), verbose=1,
)

metrics1 = evaluate_model(model1, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics1["accuracy"]}')
print(f'Precision: {metrics1["precision"]}')
print(f'Recall:    {metrics1["recall"]}')
print(f'F1-Score:  {metrics1["f1"]}')
print(f'AUC:       {metrics1["auc"]}')

plot_learning_curves(history1, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P3_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics1['accuracy'], 'precision': metrics1['precision'],
    'recall': metrics1['recall'], 'f1': metrics1['f1'], 'auc': metrics1['auc'],
    'epochs': len(history1.history['loss']), 'notes': '',
})

Epoch 1/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.8503 - loss: 0.4322

689/689 ━━━━━━━━━━━━━━━━━━━━ 5593s 8s/step - accuracy: 0.9033 - loss: 0.2738 - val_accuracy: 0.9394 - val_loss: 0.1773 - learning_rate: 0.0010
Epoch 2/10
689/689 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.9365 - loss: 0.1831

KeyboardInterrupt: 

**Interpretation**: *(How quickly did the head converge? Did frozen ImageNet features transfer well to malaria cells?)*

---
## Experiment 2: Fine-tune Last 4 Base Layers

**Hypothesis**: Unfreezing the last 4 VGG16 layers (final conv block) at a reduced LR (1e-5) allows the top features to adapt to malaria cell morphology, while keeping the earlier ImageNet features intact to prevent catastrophic forgetting.

**Change made**: Two-stage — Stage 1: frozen (10 epochs, LR=1e-3) → Stage 2: fine_tune_from=-4 (10 epochs, LR=1e-5)

In [ ]:
EXP_NUM         = 2
EXP_DESCRIPTION = 'Fine-tune last 4 base layers (LR=1e-5)'

# Stage 1: train head only
model2, base2 = build_vgg16_model(freeze_base=True)
model2.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss='binary_crossentropy', metrics=['accuracy'])
print('--- Stage 1: head only (10 epochs) ---')
history2a = model2.fit(train_ds, validation_data=val_ds, epochs=10,
                       callbacks=get_callbacks('vgg16', f'{EXP_NUM}a'), verbose=1)

# Stage 2: unfreeze last 4 base layers at lower LR
base2.trainable = True
n = len(base2.layers)
for layer in base2.layers[:n - 4]:
    layer.trainable = False
model2.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
               loss='binary_crossentropy', metrics=['accuracy'])
print('\n--- Stage 2: fine-tune last 4 layers (10 epochs, LR=1e-5) ---')
history2b = model2.fit(train_ds, validation_data=val_ds, epochs=10,
                       callbacks=get_callbacks('vgg16', EXP_NUM), verbose=1)

metrics2 = evaluate_model(model2, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics2["accuracy"]}')
print(f'Precision: {metrics2["precision"]}')
print(f'Recall:    {metrics2["recall"]}')
print(f'F1-Score:  {metrics2["f1"]}')
print(f'AUC:       {metrics2["auc"]}')

# Combine histories for plotting
combined_hist = type('H', (), {'history': {
    k: history2a.history[k] + history2b.history[k]
    for k in history2a.history
}})()
plot_learning_curves(combined_hist, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P3_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics2['accuracy'], 'precision': metrics2['precision'],
    'recall': metrics2['recall'], 'f1': metrics2['f1'], 'auc': metrics2['auc'],
    'epochs': len(history2a.history['loss']) + len(history2b.history['loss']), 'notes': '',
})

**Interpretation**: *(Did fine-tuning the last 4 layers improve over the frozen baseline? Was there a risk of overfitting in Stage 2?)*

---
## Experiment 3: Fine-tune Last 8 Base Layers

**Hypothesis**: Extending fine-tuning to the last 8 layers (two conv blocks) allows more VGG16 features to adapt to the malaria domain. This may improve performance further but increases the risk of catastrophic forgetting compared to Exp 2.

**Change made**: Two-stage — Stage 2 unfreezes last 8 base layers instead of 4

In [ ]:
EXP_NUM         = 3
EXP_DESCRIPTION = 'Fine-tune last 8 base layers (LR=1e-5)'

# Stage 1
model3, base3 = build_vgg16_model(freeze_base=True)
model3.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss='binary_crossentropy', metrics=['accuracy'])
print('--- Stage 1: head only (10 epochs) ---')
history3a = model3.fit(train_ds, validation_data=val_ds, epochs=10,
                       callbacks=get_callbacks('vgg16', f'{EXP_NUM}a'), verbose=1)

# Stage 2: unfreeze last 8 layers
base3.trainable = True
n = len(base3.layers)
for layer in base3.layers[:n - 8]:
    layer.trainable = False
model3.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
               loss='binary_crossentropy', metrics=['accuracy'])
print('\n--- Stage 2: fine-tune last 8 layers (10 epochs, LR=1e-5) ---')
history3b = model3.fit(train_ds, validation_data=val_ds, epochs=10,
                       callbacks=get_callbacks('vgg16', EXP_NUM), verbose=1)

metrics3 = evaluate_model(model3, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics3["accuracy"]}')
print(f'Precision: {metrics3["precision"]}')
print(f'Recall:    {metrics3["recall"]}')
print(f'F1-Score:  {metrics3["f1"]}')
print(f'AUC:       {metrics3["auc"]}')

combined_hist3 = type('H', (), {'history': {
    k: history3a.history[k] + history3b.history[k]
    for k in history3a.history
}})()
plot_learning_curves(combined_hist3, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P3_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics3['accuracy'], 'precision': metrics3['precision'],
    'recall': metrics3['recall'], 'f1': metrics3['f1'], 'auc': metrics3['auc'],
    'epochs': len(history3a.history['loss']) + len(history3b.history['loss']), 'notes': '',
})

**Interpretation**: *(Did 8 layers fine-tuning improve over 4 layers? Any signs of catastrophic forgetting?)*

---
## Experiment 4: Larger Classification Head (Dense 512+256)

**Hypothesis**: Increasing the head capacity to Dense(512)+Dense(256) gives the model more parameters to map VGG16 features to the binary output. Combined with frozen base, this tests whether the bottleneck lies in the head rather than the base features.

**Change made**: `dense_units=(512, 256)` with frozen base

In [ ]:
EXP_NUM         = 4
EXP_DESCRIPTION = 'Larger head Dense(512+256), frozen base'
EPOCHS          = 10

model4, base4 = build_vgg16_model(freeze_base=True, dense_units=(512, 256))
model4.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history4 = model4.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=get_callbacks('vgg16', EXP_NUM), verbose=1,
)

metrics4 = evaluate_model(model4, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics4["accuracy"]}')
print(f'Precision: {metrics4["precision"]}')
print(f'Recall:    {metrics4["recall"]}')
print(f'F1-Score:  {metrics4["f1"]}')
print(f'AUC:       {metrics4["auc"]}')

plot_learning_curves(history4, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P3_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics4['accuracy'], 'precision': metrics4['precision'],
    'recall': metrics4['recall'], 'f1': metrics4['f1'], 'auc': metrics4['auc'],
    'epochs': len(history4.history['loss']), 'notes': '',
})

**Interpretation**: *(Did the larger head improve F1 over the standard head in Exp 1? Any overfitting?)*

---
## Experiment 5: Add Data Augmentation

**Hypothesis**: Prepending data augmentation to the VGG16 pipeline increases effective training set diversity. Since VGG16 is already pretrained, augmentation should primarily benefit the head's generalisation, reducing overfitting without disrupting the base features.

**Change made**: `use_augmentation=True` with frozen base and standard head

In [ ]:
EXP_NUM         = 5
EXP_DESCRIPTION = 'Data augmentation + frozen base'
EPOCHS          = 10

model5, base5 = build_vgg16_model(freeze_base=True, use_augmentation=True)
model5.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history5 = model5.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=get_callbacks('vgg16', EXP_NUM), verbose=1,
)

metrics5 = evaluate_model(model5, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics5["accuracy"]}')
print(f'Precision: {metrics5["precision"]}')
print(f'Recall:    {metrics5["recall"]}')
print(f'F1-Score:  {metrics5["f1"]}')
print(f'AUC:       {metrics5["auc"]}')

plot_learning_curves(history5, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P3_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics5['accuracy'], 'precision': metrics5['precision'],
    'recall': metrics5['recall'], 'f1': metrics5['f1'], 'auc': metrics5['auc'],
    'epochs': len(history5.history['loss']), 'notes': '',
})

**Interpretation**: *(Did augmentation improve generalisation for a frozen VGG16 compared to Exp 1?)*

---
## Experiment 6: Lower Fine-tuning LR (LR = 1e-6)

**Hypothesis**: Using an even lower fine-tuning learning rate (1e-6 instead of 1e-5) applies smaller gradient updates to the unfrozen base layers, reducing the risk of catastrophic forgetting while still allowing feature adaptation. This may produce more stable validation curves.

**Change made**: Two-stage — Stage 2 uses `LR=1e-6` with last 4 layers unfrozen

In [ ]:
EXP_NUM         = 6
EXP_DESCRIPTION = 'Fine-tune last 4 layers at LR=1e-6'

# Stage 1
model6, base6 = build_vgg16_model(freeze_base=True)
model6.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss='binary_crossentropy', metrics=['accuracy'])
print('--- Stage 1: head only (10 epochs) ---')
history6a = model6.fit(train_ds, validation_data=val_ds, epochs=10,
                       callbacks=get_callbacks('vgg16', f'{EXP_NUM}a'), verbose=1)

# Stage 2: LR=1e-6
base6.trainable = True
n = len(base6.layers)
for layer in base6.layers[:n - 4]:
    layer.trainable = False
model6.compile(optimizer=tf.keras.optimizers.Adam(1e-6),
               loss='binary_crossentropy', metrics=['accuracy'])
print('\n--- Stage 2: fine-tune last 4 layers (10 epochs, LR=1e-6) ---')
history6b = model6.fit(train_ds, validation_data=val_ds, epochs=10,
                       callbacks=get_callbacks('vgg16', EXP_NUM), verbose=1)

metrics6 = evaluate_model(model6, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics6["accuracy"]}')
print(f'Precision: {metrics6["precision"]}')
print(f'Recall:    {metrics6["recall"]}')
print(f'F1-Score:  {metrics6["f1"]}')
print(f'AUC:       {metrics6["auc"]}')

combined_hist6 = type('H', (), {'history': {
    k: history6a.history[k] + history6b.history[k]
    for k in history6a.history
}})()
plot_learning_curves(combined_hist6, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P3_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics6['accuracy'], 'precision': metrics6['precision'],
    'recall': metrics6['recall'], 'f1': metrics6['f1'], 'auc': metrics6['auc'],
    'epochs': len(history6a.history['loss']) + len(history6b.history['loss']), 'notes': '',
})

**Interpretation**: *(Did LR=1e-6 produce more stable Stage 2 curves than LR=1e-5 in Exp 2? Better or worse final F1?)*

---
## Experiment 7: L2 Regularisation on Head

**Hypothesis**: Adding L2 weight decay (λ=1e-4) to the Dense head layers constrains the classifier's weights, reducing overfitting in the head while keeping the frozen VGG16 base unchanged. This directly targets the only trainable parameters in the frozen-base configuration.

**Change made**: `l2_lambda=1e-4` applied to all Dense layers in the head

In [ ]:
EXP_NUM         = 7
EXP_DESCRIPTION = 'L2 regularisation on head (λ=1e-4), frozen base'
EPOCHS          = 10

model7, base7 = build_vgg16_model(freeze_base=True, l2_lambda=1e-4)
model7.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history7 = model7.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=get_callbacks('vgg16', EXP_NUM), verbose=1,
)

metrics7 = evaluate_model(model7, test_ds)
print(f'\nExp {EXP_NUM} — {EXP_DESCRIPTION}')
print(f'Accuracy:  {metrics7["accuracy"]}')
print(f'Precision: {metrics7["precision"]}')
print(f'Recall:    {metrics7["recall"]}')
print(f'F1-Score:  {metrics7["f1"]}')
print(f'AUC:       {metrics7["auc"]}')

plot_learning_curves(history7, f'Exp {EXP_NUM}: {EXP_DESCRIPTION}',
                     save_path=f'figures/P3_exp{EXP_NUM}_curves.png')

results_log.append({
    'exp_num': EXP_NUM, 'description': EXP_DESCRIPTION,
    'accuracy': metrics7['accuracy'], 'precision': metrics7['precision'],
    'recall': metrics7['recall'], 'f1': metrics7['f1'], 'auc': metrics7['auc'],
    'epochs': len(history7.history['loss']), 'notes': '',
})

**Interpretation**: *(Did L2 on the head improve generalisation over no regularisation in Exp 1?)*

---
## 7. Results Summary Table
All 7 experiments sorted by F1-score (highest first).

In [ ]:
import pandas as pd
results_df = build_results_table(results_log)
pd.set_option('display.float_format', '{:.4f}'.format)
display(results_df)

## 8. Best Model — Detailed Evaluation
Identify the experiment with the highest F1-score and generate the confusion matrix, ROC curve, and error analysis.

In [ ]:
exp_map = {
    1: (model1, metrics1),
    2: (model2, metrics2),
    3: (model3, metrics3),
    4: (model4, metrics4),
    5: (model5, metrics5),
    6: (model6, metrics6),
    7: (model7, metrics7),
}

best_exp_num = results_df.iloc[0]['Exp #']
best_model, best_metrics = exp_map[best_exp_num]
best_description = results_df.iloc[0]['Description']

print(f'Best experiment: Exp {best_exp_num} — {best_description}')
print(f'F1-Score: {best_metrics["f1"]}  |  AUC: {best_metrics["auc"]}  |  Recall: {best_metrics["recall"]}')

### Confusion Matrix
Plots the confusion matrix for the best model, showing TP, TN, FP, FN counts with Sensitivity and Specificity annotated for clinical context.

In [ ]:
plot_confusion_matrix(
    best_metrics, CLASS_NAMES,
    f'Best Model (Exp {best_exp_num}): {best_description}',
    save_path='figures/P3_best_confusion_matrix.png',
)

### ROC Curve
Plots the ROC curve for the best model. AUC summarises discriminative ability across all classification thresholds — essential for comparing transfer learning models against custom CNNs.

In [ ]:
plot_roc_curve(
    best_metrics,
    f'Best Model (Exp {best_exp_num}): {best_description}',
    save_path='figures/P3_best_roc_curve.png',
)

### Error Analysis
Displays misclassified test images to identify morphological patterns the VGG16 features struggle to distinguish.

In [ ]:
error_analysis(best_model, test_ds, CLASS_NAMES, n_samples=12)

## 9. Model Summary & Report Notes

### Best configuration
- **Experiment**: *(fill in after running)*
- **Architecture**: VGG16 (ImageNet) + GAP + Dense head + sigmoid
- **Training strategy**: *(frozen only / two-stage fine-tune)*
- **Key hyperparameters**: *(LR stage 1, LR stage 2, epochs, regularisation)*
- **Test metrics**: Accuracy=X, Precision=X, Recall=X, F1=X, AUC=X

### Clinical relevance
*(Discuss Recall/Sensitivity — does VGG16's pretrained feature extraction improve false-negative rate compared to the baseline CNN? Clinically, which matters more here?)*

### Observed patterns
- **Frozen vs fine-tuned**: *(Did fine-tuning consistently improve over frozen?)*
- **Optimal fine-tune depth**: *(4 layers vs 8 layers — which was better?)*
- **Most impactful change**: *(Which experiment produced the largest F1 gain?)*
- **Transfer learning effectiveness**: *(Did ImageNet features transfer well to malaria cells?)*

### Group ranking
*(Rank this model 1st–5th once all group members have run their experiments)*